# Genotype PCA

This notebook computes **principal components (PCs) from the genotype data** to use as covariates that correct for population structure in downstream QTL analysis.


## Description

This notebook computes **principal components (PCs)** from the genotype data to use as covariates that correct for population structure in downstream QTL analysis.

To estimate genotype PCs accurately we account for **relatedness** between samples, because closely related individuals share large genomic segments that distort the principal-component space. We use **KING** to estimate kinship and, when relatives exist, split the samples into an **unrelated** set and a **related** set. The PCA model is fit on the unrelated set, and the related samples are then **projected** onto that model so every sample ends up with PC scores in the same space.

> **Note on this toy dataset:** the 60-sample example contains **one related pair** (SAMPLE_059 and SAMPLE_060 are a 1st-degree, parent-offspring pair with kinship ~0.25). KING therefore splits the cohort into **59 unrelated** samples and **1 related** sample (SAMPLE_060). Steps 3-4 fit the PCA model on the 59 unrelated samples; Steps 5-6 project SAMPLE_060 onto that model.

## Input Files

| File | Description |
| --- | --- |
| `output/plink/protocol_example.genotype.merged.plink_qc.{bed,bim,fam}` | QC'd, merged genotype in PLINK format (from `2_genotype_preprocessing.ipynb`). |
| `output/rnaseq/protocol_example.rnaseq.bed.bed.gz` | Phenotype bed (from `1_phenotype_preprocessing.ipynb`); used to define the set of samples that have both genotype and phenotype. |


## Steps


### **Step 1.** Match genotype samples to phenotype samples

Keep only the samples present in **both** the genotype and the phenotype. The resulting `*.sample_genotypes.txt` is the keep-list used by KING in the next step.


In [ ]:
sos run pipeline/GWAS_QC.ipynb genotype_phenotype_sample_overlap \
    --cwd output/genotype/ \
    --genoFile output/plink/protocol_example.genotype.merged.plink_qc.fam \
    --phenoFile output/rnaseq/protocol_example.rnaseq.bed.bed.gz


### **Step 2.** Estimate kinship (KING)

KING estimates pairwise kinship coefficients and, if relatives are found, splits the cohort into an **unrelated** set (`*.king.unrelated.bed`) and a **related** set (`*.king.related.bed`). The `.kin0` file lists kinship coefficients for related pairs.

On this toy dataset KING detects the related pair **SAMPLE_059 - SAMPLE_060** (kinship ~0.25, IBS0 = 0, i.e. parent-offspring). It writes `*.king.related.bed` (1 sample: SAMPLE_060) and `*.king.unrelated.bed` (59 samples), and we proceed with the unrelated set in Step 3.

In [ ]:
sos run pipeline/GWAS_QC.ipynb king \
    --cwd output/genotype/kinship \
    --genoFile output/plink/protocol_example.genotype.merged.plink_qc.bed \
    --name protocol_example.genotype.king \
    --keep-samples output/genotype/protocol_example.rnaseq.bed.bed.sample_genotypes.txt


### **Step 3.** QC + LD pruning on the unrelated set

We apply variant- and sample-level QC (here filtering minor-allele count < 5) and **LD-prune the 59 unrelated samples** (`*.king.unrelated.bed`) in preparation for PCA. LD pruning removes highly correlated SNPs so the PCs reflect genome-wide structure. The pruned set (`*.prune.bed`) and retained-SNP list (`*.prune.in`) are written under the working directory.

> **LD-pruning sample-size requirement:** plink2 `--indep-pairwise` LD estimator **requires at least 50 samples**; with 59 unrelated samples this runs cleanly.

In [ ]:
# QC + LD pruning on the 59 unrelated samples (KING unrelated set)
sos run pipeline/GWAS_QC.ipynb qc \
    --cwd output/genotype/genotype \
    --genoFile output/genotype/kinship/protocol_example.genotype.merged.plink_qc.protocol_example.genotype.king.unrelated.bed \
    --mac-filter 5 -s force

### **Step 4.** flashPCA on the unrelated set

Run `flashpca` on the pruned unrelated genotype. This fits the PCA model, computes Mahalanobis distances to flag outliers, and plots PC scatter and scree plot. These genotype PCs are the population-structure covariates used in the QTL analysis.

In [ ]:
sos run pipeline/PCA.ipynb flashpca \
    --cwd output/genotype/genotype_pca \
    --genoFile output/genotype/genotype/protocol_example.genotype.merged.plink_qc.protocol_example.genotype.king.unrelated.plink_qc.prune.bed

### **Step 5.** QC the related set (keep unrelated-PCA SNPs)

QC the related samples **without** LD pruning, restricted to the same SNPs retained for the unrelated PCA (`--keep-variants ...prune.in`), so the related genotypes are on the identical SNP set as the PCA model.

In [ ]:
K=output/genotype/kinship/protocol_example.genotype.merged.plink_qc.protocol_example.genotype.king
sos run pipeline/GWAS_QC.ipynb qc_no_prune \
    --cwd output/genotype/genotype \
    --genoFile ${K}.related.bed \
    --maf-filter 0 --geno-filter 0 --mind-filter 0.1 --hwe-filter 0 \
    --keep-variants output/genotype/genotype/protocol_example.genotype.merged.plink_qc.protocol_example.genotype.king.unrelated.plink_qc.prune.in

### **Step 6.** Project the related sample onto the unrelated PCA model

Project the related sample (SAMPLE_060) onto the unrelated PCA model with `PCA.ipynb project_samples`, so every sample ends up with PC scores in the same space.

In [ ]:
sos run pipeline/PCA.ipynb project_samples \
    --cwd output/genotype/genotype_pca \
    --genoFile output/genotype/genotype/protocol_example.genotype.merged.plink_qc.protocol_example.genotype.king.related.plink_qc.extracted.bed \
    --pca-model output/genotype/genotype_pca/protocol_example.genotype.merged.plink_qc.protocol_example.genotype.king.unrelated.plink_qc.prune.pca.rds

# PCA on genotypes of selected samples

This notebook contains workflow to compute PCA-derived covariates from the genotype data.

## Methods overview

This workflow is an application of `PCA.ipynb` from the xQTL project pipeline.

## Data Input

- `output/plink/wgs.merged.plink_qc.bed`
- `output/plink/wgs.merged.plink_qc.bim`
- `output/plink/wgs.merged.plink_qc.fam`

## Data Output
- no related samples: `output/genotype/genotype_pca/wgs.merged.plink_qc.plink_qc.prune.pca.rds`
- with related samples: `output/genotype/genotype_pca/wgs.merged.plink_qc.wgs.merged.king.related.plink_qc.extracted.pca.projected.rds`


## Steps in detail

### Kinship QC only on proteomics samples

To accuratly estimate the PCs for the genotype. We split participants based on their kinship coefficients, estimated by KING

#### Sample match with genotype 
-- `Aim`: In this chunk, we only want to keep the samples in genotype overlapped with phenotype to do king estimation. sample_genotypes.txt would be used as a keep sample list in the next `king` chunk after `genotype_phenotype_sample_overlap` .

-- `Main input`: 
- phenofile: should be the bed.gz file in the output of penotype preprocessing.   
- genofile: should be the output of genotype preprocessing.

-- `Output`:    
sample_overlap.txt, sample_genotypes.txt.    
These outputs are sample list of genotype overlapped with phenotype.    

#### Kinship
`[king_1]`:   
-- `Aim`: it is designed to infer relationships within a sample set to identify closely related individuals.   
-- `Main input`: plink genofile, kin_maf: A parameter that specifies the minor allele frequency to filter SNPs. The --keep and --remove options might be used if the keep_samples and remove_samples files are provided. These options allow for including or excluding specific samples.  
-- `Output`: The primary output is a .kin0 file, which contains the kinship coefficients for pairs of individuals. A higher kinship coefficient indicates a closer genetic relationship between two individuals. This file helps in identifying closely related individuals.  

`[king_2]`:   
-- `Aim`: To select a list of unrelated individuals from the data. The goal is to maximize the number of unrelated individuals selected while filtering out those who are related. This is useful in genetic studies where relatedness can confound results.   
-- `Main input`: a .kin0 file containing kinship coefficients for pairs of individuals. maximize_unrelated: A boolean parameter that determines whether the workflow should attempt to maximize the number of unrelated individuals. True for keeping as many unrelated individuals as possible, False for removing entire families with any related individuals.     
-- `Output`:  a file with the extension .related_id, which contains a list of related individuals that should be excluded from further analysis.   

`[king_3]`:   
-- `Aim`: To split genotype data into two sets: one containing unrelated samples and the other containing related samples.   
-- `Main input`: output_from(2): This input is the output from the previous step (presumably king_2), which should contain the list of related individuals. genoFile: This is the primary genotype data file that will be split based on relatedness.
-- `Output`: unrelated_bed: This is the output file containing genotype data for unrelated individuals. related_bed: This is the output file containing genotype data for related individuals.

`In summary`, the `king` workflows provide a comprehensive approach to handle relatedness in genotype data. Starting from identifying related individuals, to selecting a set of unrelated samples, and finally splitting the data based on relatedness, these workflows ensure that genetic analyses can be conducted on appropriately filtered datasets.

related result is shown below:

**Columns Explanation**:  
-- FID1 & IID1: Family and individual identifiers for the first sample.  
-- FID2 & IID2: Family and individual identifiers for the second sample.  
-- NSNP: The number of SNPs (Single Nucleotide Polymorphisms) that the two samples share.  
-- HETHET: The proportion of SNPs where both samples are heterozygous.  
-- IBS0: The proportion of SNPs where the two samples have two different alleles.  
-- KINSHIP: The kinship coefficient, indicating the genetic relationship between the two samples.  

Variant level and sample level QC on unrelated individuals using missingness > 10%, and LD-prunning in preparation for PCA analysis.    


**Be aware:**    

**If the message from `king_2` shown as `No related individuals detected from *.kin0`, this means no related individuals detected for the samples in `--keep_samples`. In this case, there will be no output for unrelated individuals from this step.**

**In other cases eg ROSMAP proteomics data, message `No related individuals detected from *.kin0` occured, there is no separate genotype data generated for unrelated individuals. In this case, we need to work from the original genotype data and must use `--keep-samples` to run `qc` to extract samples for PCA.**

#### QC on unrelated samples


Here we write data to `cache` folder instead of `output` because this genotype data can be removed later after PCA. Also filter out minor allel accout < 5.

**If your data has `*.unrelated.bed` generated, that means there are related individuals in your data. In cases, we will use output from the KING step for unrelated individuals.**

About `qc`:   
1. `[qc_no_prune, qc_1 (basic QC filters)]`:  
-- `aim`: To filter SNPs and select individuals based on various quality control (QC) criteria. The goal is to ensure that the genotype data is of high quality and free from potential errors or biases before further analysis.   
-- `Input`:   
genoFile: The primary input file containing genotype data.  
Various parameters that dictate the QC criteria:  
maf_filter, maf_max_filter: Minimum and maximum Minor Allele Frequency (MAF) thresholds.  
mac_filter, mac_max_filter: Minimum and maximum Minor Allele Count (MAC) thresholds.  
geno_filter: Maximum missingness per variant.  
mind_filter: Maximum missingness per sample.  
hwe_filter: Hardy-Weinberg Equilibrium (HWE) filter threshold.  
other_args: Other optional PLINK arguments.  
meta_only: Flag to determine if only SNP and sample lists should be output.  
rm_dups: Flag to remove duplicate variants.  
-- `Output`: A file (or set of files) with the suffix .plink_qc (and possibly .extracted if specific variants are kept). The exact format (e.g., .bed or .snplist) depends on the meta_only parameter.  

2. [qc_2 (LD pruning)]:   
-- `aim`: To perform Linkage Disequilibrium (LD) pruning and remove related individuals (both individuals of a pair). LD pruning is a common step in genotype data quality control, aiming to remove highly correlated SNPs, thus reducing redundancy in the data and enhancing the accuracy of subsequent analyses.   
-- `Input`:
_input: The primary input file containing genotype data that has undergone basic quality control.   
Pruning parameters:   
window: The window size for calculating LD between SNPs.   
shift: The number of SNPs to shift the window each time.   
r2: The LD threshold for pruning   
-- `Output`:  
.prune.bed: The binary PLINK format file of the pruned genotype data.   
.prune.in: A list containing the SNPs to retain.

if there are unrelated data & related data, treat them separately

#### QC on related samples

#### PCA on unrelated samples
Note PC1 vs 2 outlier

About `[flashpca]`:   
1. `[flashpca_1]`:     
-- `aim`: To perform Principal Component Analysis (PCA) on genotype data using the flashpcaR library. PCA is a statistical method used to emphasize variation and bring out strong patterns in a dataset. In the context of genomics, PCA is often used to identify and correct for population stratification in genome-wide association studies.   
-- `Input`:    
genoFile: A binary PLINK file containing genotype data after qc.    
Various parameters for PCA and data filtering, such as min_pop_size, stand, and others.   
-- `Output`:    
.pca.rds: An RDS file containing the PCA results, including the PCA model, scores, and metadata.    
.txt: A text file containing the PCA scores for each individual.   

2. `[flashpca_2, project_samples_2]`:   Outlier Detection   
-- `aim`: To detect outliers based on Mahalanobis distance, which measures the distance of a point from a distribution.     
-- `Input`:  pca result     
-- `Output`:         
distance: An RDS file containing Mahalanobis distances for each sample.    
identified_outliers: A file listing the identified outliers.    
analysis_summary: A markdown file summarizing the analysis.   
qqplot_mahalanobis: A QQ plot visualizing the Mahalanobis distances.    
hist_mahalanobis: A histogram of the Mahalanobis distances.    

3. `[flashpca_3, project_samples_3]`: PCA Visualization    
-- `aim`: To visualize the PCA results, highlighting any identified outliers.   
-- `Input`:  
PCA results from the previous step.    
List of identified outliers.   
-- `Output`:    
PCA plot (*.pc.png): A scatter plot of 2 adjacent principal components, with outliers highlighted.    
Scree plot (*.scree.png): A plot showing the variance explained by each principal component.    

#### Project PCA results back to related samples

The workflow aims to project the PCA results of unrelated samples onto the related samples. This is useful because PCA is typically performed on unrelated samples to avoid the confounding effects of relatedness. This is often done to ensure that related samples are analyzed in the same "space" as the unrelated samples, making the results more comparable and interpretable. Once the primary PCA model is established with unrelated samples, the related samples can be projected onto this model to obtain their principal component scores.



# the final pca output that we will use in cov processing
`related.plink_qc.extracted.pca.projected.rds`

## Output Files

| File | Description |
| --- | --- |
| `output/genotype/protocol_example.rnaseq.bed.bed.sample_genotypes.txt` | Samples present in both genotype and phenotype (keep-list). |
| `output/genotype/kinship/*.king.kin0` | KING kinship table; lists the related pair SAMPLE_059 - SAMPLE_060. |
| `output/genotype/kinship/*.king.unrelated.bed` / `*.king.related.bed` | Genotype split into 59 unrelated and 1 related (SAMPLE_060) samples. |
| `output/genotype/genotype/*.king.unrelated.plink_qc.prune.{bed,in}` | QC + LD-pruned unrelated genotype and the retained-SNP list. |
| `output/genotype/genotype_pca/*.king.unrelated.plink_qc.prune.pca.rds` | Fitted flashPCA model (PCs for the 59 unrelated samples). |
| `output/genotype/genotype/*.king.related.plink_qc.extracted.bed` | Related sample restricted to the unrelated-PCA SNP set. |
| `output/genotype/genotype_pca/*.extracted.pca.projected.*` | Related sample (SAMPLE_060) projected onto the unrelated PCA model (`.pc.png`, `.scree.png`, `.mahalanobis.rds`). |

## Anticipated Results

KING detects **one related pair** among the 60 samples (SAMPLE_059 - SAMPLE_060, kinship ~0.25, IBS0 = 0 -> parent-offspring) and splits the cohort into **59 unrelated** and **1 related** sample (SAMPLE_060). QC + LD pruning and flashPCA run on the **59 unrelated samples** to fit the PCA model (`*.unrelated.plink_qc.prune.pca.rds`), producing PC scatter / scree plots and a Mahalanobis outlier table. The related sample is then QC-restricted to the unrelated-PCA SNP set and **projected** onto that model (`*.extracted.pca.projected.*`), so all 60 samples have genotype PCs in the same space for use as covariates in the QTL analysis.